In [ ]:
from circkit.bitwise import BitwiseCircuit
from circkit.arithmetic import ArithmeticCircuit
from circkit import Circuit, Operation, Param

In [41]:
def EvenBits(word_size):
    Table = [0]*2^word_size
    power = [1]*word_size
    
    for i in range(1,word_size):
        power[i] = 4*power[i-1]
    
    for i in range(2^word_size):
        b = Integer(i).digits(2)
        Table[i] = sum([power[j]*b[j] for j in range(len(b))])
    return Table

In [42]:
def dict_of_list_to_list(dictionnary):
    L = []
    for x in dictionnary.values():
        L += x
    return set(L)

In [43]:
def fun_to_circuit_BitWise(fun,nb_inputs,word_size,is_input_a_list = False):
    C = BitwiseCircuit(name=fun.__name__,word_size=word_size)
    inp_nodes = C.add_inputs(n=nb_inputs, format="x_%d")
    if(is_input_a_list):
        C.add_output(fun(inp_nodes))
    else :
        C.add_output(fun(*inp_nodes))
    return C

In [44]:
def get_inputs_node(node,var_calc):
    
    a = node.incoming[0]
        
    #a_id_BitW is the index of the left input gate of the sum gate in C_BitW
    a_id_BitW = a.id

    #a is the variable in C_Arith linked to the gate which is the left input of the sum
    a = var_calc[a_id_BitW]

    b = None
    if len(node.incoming) >1 :
        b = node.incoming[1]
        #b_id_BitW is the index of the right input gate of the sum gate in C_BitW
        b_id_BitW = b.id

        #b is the variable in C_Arith linked to the gate which is the right input of the sum
        b = var_calc[b_id_BitW]

    return a,b

In [1]:
def get_gate_to_LK(C_Arith,cpt_w,dict_decomp):
    
    c_e = C_Arith.add_input("w_%d"%cpt_w)
    cpt_w += 1
    
    c_o = C_Arith.add_input("w_%d"%cpt_w)
    cpt_w += 1
    
    c = 2*c_o + c_e
    
    dict_decomp[c] = (c_e,c_o)
    return c,cpt_w

In [46]:
def inp_BitW_to_Arith(C_Arith,cpt_x,cpt_w,dict_decomp):
    
    inp = C_Arith.add_input("x_%d"%cpt_x)
    cpt_x += 1
    
    potential_inp,cpt_w = get_gate_to_LK(C_Arith,cpt_w,dict_decomp)
    
    verif_nodes_are_equal = potential_inp - inp
    
    return potential_inp,[verif_nodes_are_equal],cpt_x,cpt_w

In [25]:
def add_cst_Arith(C_Arith,cst,word_size,dict_decomp, already_compute):
    
    if cst in already_compute :       
        return already_compute[cst]
    
    v_cst_e,v_cst_o = decomp(cst,word_size,{})
    cst_e = C_Arith.add_const(v_cst_e)

    if v_cst_e == v_cst_o:
        cst_o = cst_e
    else:
        cst_o = C_Arith.add_const(v_cst_o)

    c = 2*cst_o + cst_e

    already_compute[cst] = c
    dict_decomp[c] = (cst_e,cst_o)
    
    return c

In [1]:
def already_compute_variable(C_Arith,a,b,cpt_w,dict_decomp,already_compute,new_variable=True):
    #print(a,b)
    inp_of_gate = (a,b)
    if b == None :
        inp_of_gate = a
    #print(already_compute)
    if inp_of_gate in already_compute :
        return True,already_compute[inp_of_gate],cpt_w
    
    c = None
    if new_variable :
        c,cpt_w = get_gate_to_LK(C_Arith,cpt_w,dict_decomp)
        already_compute[inp_of_gate] = c
    
    return False,c,cpt_w

In [47]:
def common_party_XOR_AND(C_Arith,a,b,cpt_w,dict_decomp):
    a_e,a_o = dict_decomp[a]
    b_e,b_o = dict_decomp[b]
    
    if (a_e,b_e) in dict_decomp :
        e_e,e_o = dict_decomp[(a_e,b_e)]
        o_e,o_o = dict_decomp[(a_o,b_o)]
        #print("We already have prove the XOR/AND/OR")
        return [], e_e,e_o,o_e,o_o, cpt_w,False
        
    e_e = C_Arith.add_input("w_%d"%cpt_w)
    cpt_w += 1
    e_o = C_Arith.add_input("w_%d"%cpt_w)
    cpt_w += 1
    
    o_e = C_Arith.add_input("w_%d"%cpt_w)
    cpt_w += 1
    o_o = C_Arith.add_input("w_%d"%cpt_w)
    cpt_w += 1

    
    
    dict_decomp[(a_e,b_e)] = [e_e,e_o]
    e = a_e + b_e
    is_e_right_evaluated = e - (2*e_o + e_e)
    
    
    
    dict_decomp[(a_o,b_o)] = [o_e,o_o]
    o = a_o + b_o
    is_o_right_evaluated = o - (2*o_o + o_e)
    
    return [is_e_right_evaluated,is_o_right_evaluated], e_e,e_o,o_e,o_o, cpt_w,True

In [48]:
def AND_BitW_to_Arith(C_Arith,a,b,cpt_w,dict_decomp,already_compute):
    #print("AND")
    c_already_compute,c,cpt_w = already_compute_variable(C_Arith,a,b,cpt_w,dict_decomp,already_compute,new_variable=False)
    if c_already_compute :
        #print("The AND gate is already computed, no proof to provide")
        return c,[],cpt_w,False
    
    to_verif, e_e,e_o,o_e,o_o, cpt_w, need_proof = common_party_XOR_AND(C_Arith,a,b,cpt_w,dict_decomp)
    
    c = 2*o_o + e_o
    dict_decomp[c]=(e_o,o_o)
    
    already_compute[(a,b)] = c
    
    return c,to_verif,cpt_w,need_proof

In [49]:
def XOR_BitW_to_Arith(C_Arith,a,b,cpt_w,dict_decomp,already_compute):
    #print("XOR")
    c_already_compute,c,cpt_w = already_compute_variable(C_Arith,a,b,cpt_w,dict_decomp,already_compute,new_variable=False)
    if c_already_compute :
        #print("The XOR gate is already computed, no proof to provide")
        #print(a,b)
        #print("\n")
        return c,[],cpt_w,False
    
    to_verif, e_e,e_o,o_e,o_o, cpt_w, need_proof = common_party_XOR_AND(C_Arith,a,b,cpt_w,dict_decomp)
    
    c = 2*o_e + e_e
    dict_decomp[c]=(e_e,o_e)
    
    already_compute[(a,b)] = c
    
    return c,to_verif,cpt_w,need_proof

In [50]:
def OR_BitW_to_Arith(C_Arith,a,b,cpt_w,dict_decomp,already_compute_OR,already_compute_XOR,already_compute_AND):
    #print("OR")
    c_already_compute,c,cpt_w = already_compute_variable(C_Arith,a,b,cpt_w,dict_decomp,already_compute_OR,new_variable=False)
    if c_already_compute :
        #print("The OR gate is already computed, no proof to provide")
        return c,[],cpt_w,False
    
    c_XOR, to_verif_XOR, cpt_w,need_proof = XOR_BitW_to_Arith(C_Arith,a,b,cpt_w,dict_decomp,already_compute_XOR)
    
    c_AND, to_verif_AND, cpt_w,_ = AND_BitW_to_Arith(C_Arith,a,b,cpt_w,dict_decomp,already_compute_AND)
    
    c = c_XOR + c_AND
    
    e_e,o_e = dict_decomp[c_XOR]
    e_o,o_o = dict_decomp[c_AND]
    dict_decomp[c] = (e_e+e_o,o_e+o_o)
    
    already_compute_OR[(a,b)] = c
    
    return c,to_verif_XOR+to_verif_AND,cpt_w,need_proof

In [51]:
def NOT_BitW_to_Arith(C_Arith,a,cpt_w,word_size,dict_decomp,dict_cst,already_compute_NOT,already_compute_XOR):
    #print("NOT")
    c_already_compute,c,cpt_w = already_compute_variable(C_Arith,a,None,cpt_w,dict_decomp,already_compute_NOT,new_variable=False)
    if c_already_compute :
        #print("The NOT gate is already computed, no proof to provide")
        return c,[],cpt_w,False
    
    cst_max = add_cst_Arith(C_Arith,2^word_size-1,word_size,dict_decomp,dict_cst)
    c,to_verif,cpt_w,need_proof = XOR_BitW_to_Arith(C_Arith,a,cst_max,cpt_w,dict_decomp,{})
    
    already_compute_NOT[a] = c
    return c,to_verif,cpt_w,need_proof

In [52]:
def ADD_Verif(C_Arith,a,b,c,cpt_w,word_size):
    
    add = a + b
    
    k = C_Arith.add_input("w_%d"%cpt_w)
    cpt_w += 1
    
    check_k_in_0_1 =  k * (k-1)
    
    is_right_evaluated = add - (c + k*2^word_size)
    
    return [is_right_evaluated,check_k_in_0_1],cpt_w

In [ ]:
def ADD_BitW_to_Arith(C_Arith,a,b,cpt_w,word_size,dict_decomp,already_compute):
    #print("ADD")
    c_already_compute,c,cpt_w = already_compute_variable(C_Arith,a,b,cpt_w,dict_decomp,already_compute)
    if c_already_compute :
        #print("The ADD gate is already computed, no proof to provide")
        return c,[],cpt_w,False
    
    return c,*ADD_Verif(C_Arith,a,b,c,cpt_w,word_size),True

In [53]:
def SUB_BitW_to_Arith(C_Arith,a,b,cpt_w,word_size,dict_decomp,already_compute):
    #print("SUB")
    c_already_compute,c,cpt_w = already_compute_variable(C_Arith,a,b,cpt_w,dict_decomp,already_compute)
    if c_already_compute :
        #print("The SUB gate is already computed, no proof to provide")
        return c,[],cpt_w,False
    
    return c,*ADD_Verif(C_Arith,b,c,a,cpt_w,word_size),True

In [54]:
def NEG_BitW_to_Arith(C_Arith,a,cpt_w,word_size,dict_decomp,dict_cst,already_compute):
    
    zero = add_cst_Arith(C_Arith,0,word_size,dict_decomp,dict_cst)
    
    c_already_compute,c,cpt_w = already_compute_variable(C_Arith,a,zero,cpt_w,dict_decomp,already_compute)
    if c_already_compute :
        #print("The NEG gate is already computed, no proof to provide")
        return c,[],cpt_w,False
    
    return c,*ADD_Verif(C_Arith,a,c,zero,cpt_w,word_size),True

In [55]:
def MUL_Verif(C_Arith,a,b,c,cpt_w,word_size,dict_decomp):
    
    mul = a * b
    
    k,cpt_w = get_gate_to_LK(C_Arith,cpt_w,dict_decomp)
    
    is_right_evaluated = mul - (c + k*2^word_size)
    
    return [is_right_evaluated],cpt_w

In [ ]:
def MUL_BitW_to_Arith(C_Arith,a,b,cpt_w,word_size,dict_decomp,already_compute):
    #print("MUL")
    c_already_compute,c,cpt_w = already_compute_variable(C_Arith,a,b,cpt_w,dict_decomp,already_compute)
    if c_already_compute :
        #print("The MUL gate is already computed, no proof to provide")
        return c,[],cpt_w,False
    
    return c,*MUL_Verif(C_Arith,a,b,c,cpt_w,word_size,dict_decomp),True

In [56]:
def common_party_DIV_MOD(C_Arith,a,b,cpt_w,dict_decomp,already_compute):
    
    if (a,b) in already_compute:
        return [],*already_compute[(a,b)],cpt_w,False
    
    q,cpt_w = get_gate_to_LK(C_Arith,cpt_w,dict_decomp)
    r,cpt_w = get_gate_to_LK(C_Arith,cpt_w,dict_decomp)
    
    already_compute[(a,b)] = [q,r]
    
    is_right_evaluated = a - (b*q + r)
    
    b_min_r_min_1,cpt_w = get_gate_to_LK(C_Arith,cpt_w,dict_decomp)
    
    is_r_less_than_b = b-r-1 - b_min_r_min_1
    
    return [is_right_evaluated, is_r_less_than_b],q,r,cpt_w,True

In [57]:
def DIV_BitW_to_Arith(C_Arith,a,b,cpt_w,dict_decomp,already_compute):
    
    to_verif, q,r, cpt_w, need_proof = common_party_DIV_MOD(C_Arith,a,b,cpt_w,dict_decomp,already_compute)
    
    return q,to_verif,cpt_w,need_proof

In [58]:
def MOD_BitW_to_Arith(C_Arith,a,b,cpt_w,dict_decomp,already_compute):
    
    to_verif, q,r, cpt_w,need_proof = common_party_DIV_MOD(C_Arith,a,b,cpt_w,dict_decomp,already_compute)
    
    return r,to_verif,cpt_w,need_proof

In [ ]:
def SHL_BitW_to_Arith(C_Arith,a,b,cpt_w,word_size,dict_decomp,already_compute_cst,already_compute_SHL,already_compute_MUL):
    if b >= word_size:
        #print("The offset is to big, we replace the SHL gate by 0")
        zero = add_cst_Arith(C_Arith,0,word_size,dict_decomp, already_compute_cst)
        return zero,[],cpt_w,False
    
    c_already_compute,c,cpt_w = already_compute_variable(C_Arith,a,b,cpt_w,dict_decomp,already_compute_SHL,new_variable=False)
    if c_already_compute :
        #print("The SHL gate is already computed, no proof to provide")
        return c,[],cpt_w,False
    
    two_pow_b = add_cst_Arith(C_Arith,2^b,word_size,dict_decomp, already_compute_cst)

    return MUL_BitW_to_Arith(C_Arith,a,two_pow_b,cpt_w,word_size,dict_decomp,already_compute_MUL)

In [ ]:
def SHR_BitW_to_Arith(C_Arith,a,b,cpt_w,word_size,dict_decomp,already_compute_cst,already_compute_SHR,already_compute_DIV):
    if b >= word_size:
        #print("The offset is to big, we replace the SHR gate by 0")
        zero = add_cst_Arith(C_Arith,0,word_size,dict_decomp, already_compute_cst)
        return zero,[],cpt_w,False
        
    c_already_compute,c,cpt_w = already_compute_variable(C_Arith,a,b,cpt_w,dict_decomp,already_compute_SHR,new_variable=False)
    if c_already_compute :
        #print("The SHR gate is already computed, no proof to provide")
        return c,[],cpt_w,False
    
    two_pow_b = add_cst_Arith(C_Arith,2^b,word_size,dict_decomp, already_compute_cst)

    return DIV_BitW_to_Arith(C_Arith,a,two_pow_b,cpt_w,dict_decomp,already_compute_DIV)

In [ ]:
def SHL_BitW_to_Arith_first_version(C_Arith,a,b,c,cpt_w,word_size,dict_decomp,list_to_pow):
    
    equal_1_if_b_greater_than_word_size = C_Arith.add_input("w_%d"%cpt_w)
    cpt_w += 1
    
    check_equal_1_if_b_greater_than_word_size_in_0_1 =  equal_1_if_b_greater_than_word_size * (equal_1_if_b_greater_than_word_size-1)
    
    word_size_minus_b,cpt_w = get_gate_to_LK(C_Arith,cpt_w,dict_decomp)
    
    check_word_size_minus_b_minus_one_right_computed = word_size - b - 1 - word_size_minus_b
    
    check_equal_1_if_b_greater_than_word_size_right_computed = (equal_1_if_b_greater_than_word_size-1)*check_word_size_minus_b_minus_one_right_computed
    
    claimed_2_pow_b_mod_2_pow_word_size = C_Arith.add_input("w_%d"%cpt_w)
    cpt_w += 1

    list_to_pow += [b+equal_1_if_b_greater_than_word_size*(word_size-b),claimed_2_pow_b_mod_2_pow_word_size]
    
    to_verif,cpt_w = MUL_BitW_to_Arith(C_Arith,a,claimed_2_pow_b_mod_2_pow_word_size,c,cpt_w,word_size,dict_decomp)
    
    return to_verif+[check_equal_1_if_b_greater_than_word_size_in_0_1,check_equal_1_if_b_greater_than_word_size_right_computed],cpt_w